# Airport Connectivity Analysis - Assignment 3

This notebook uses reusable utilities from `utils.py` so you can swap datasets by changing only the input paths/schema config.

Visualizations replicate Assignment 2 analyses using **Bokeh** with map-style coordinates, directed arrows, and linked edge highlighting on node hover/select.

In [12]:
from pathlib import Path
import importlib
from IPython.display import display
from bokeh.io import output_notebook, show

import utils
importlib.reload(utils)

from utils import (
    AirportDatasetConfig,
    run_pipeline,
    plot_all_connections,
    plot_one_way_connections,
    plot_one_way_degree_diff,
    plot_two_way_connections,
    plot_centrality_grid,
    plot_degree_histogram,
    plot_local_clustering_map,
    plot_center_periphery,
    plot_degree_distribution_loglog,
    louvain_communities_plot,
    leiden_communities_plot,
    girvan_newman_communities_plot,
    louvain_communities_map_plot,
    plot_louvain_coassignment_heatmaps_pair,
    plot_louvain_community_mean_degrees,
    plot_louvain_community_node_counts,
    plot_louvain_modularity_trajectory,
    louvain_reference_partition,
    louvain_modularity_by_level,
)

output_notebook()

Loading BokehJS ...

In [13]:
# Change only this block to use another airport dataset
DATA_ROOT = Path('../data')

config = AirportDatasetConfig(
    nodes_path=str(DATA_ROOT / 'reachability-meta.csv' / 'reachability-meta.csv'),
    edges_path=str(DATA_ROOT / 'reachability.txt' / 'reachability.txt'),
    node_id_col='node_id',
    node_name_col='name',
    node_lat_col='latitude',
    node_lon_col='longitude',
    node_pop_col='metro_pop',
    edge_source_col='FromNodeId',
    edge_target_col='ToNodeId',
    edge_weight_col='Weight',
)

In [14]:
result = run_pipeline(config)

G = result['G']
G_one_way = result['G_one_way']
G_undirected = result['G_undirected']

print('Directed summary:')
display(result['directed_table'])

print('Undirected summary:')
display(result['undirected_table'])

Directed summary:


,nodes,edges,scc_count,largest_scc_size
0,456,71959,1,456


Undirected summary:


,nodes,edges,avg_degree,density,connected_components,largest_cc_size,avg_clustering,transitivity,mean_shortest_path,diameter,radius
0,456,34012,149.175439,0.327858,1,456,0.806788,0.590841,1.674745,3,2


In [ ]:
show(plot_all_connections(G, max_edges=600))
show(plot_one_way_connections(G_one_way, max_edges=600))
show(plot_one_way_degree_diff(G_one_way, max_edges=600))
show(plot_two_way_connections(G_undirected, max_edges=600))

In [16]:
centrality_grid = plot_centrality_grid(G_undirected, max_edges_each=700)
show(centrality_grid)

In [17]:
show(plot_degree_histogram(G_undirected))
show(plot_local_clustering_map(G_undirected, max_edges=900))
show(plot_center_periphery(G_undirected, max_edges=900))
show(plot_degree_distribution_loglog(G_undirected, add_fit=False))

## Community Detection Comparison
This section compares Louvain, Leiden, and Girvan-Newman communities on the undirected airport graph.

In [18]:
louvain_fig, louvain_modularity = louvain_communities_plot(G_undirected, max_edges=900)
print(f'Louvain modularity: {louvain_modularity:.4f}')
show(louvain_fig)

leiden_fig, leiden_modularity = leiden_communities_plot(G_undirected, max_edges=900)
print(f'Leiden modularity: {leiden_modularity:.4f}')
show(leiden_fig)

# girvan_fig, girvan_modularity = girvan_newman_communities_plot(G_undirected, max_edges=900, max_levels=6)
# print(f'Girvan-Newman modularity: {girvan_modularity:.4f}')
# show(girvan_fig)

Louvain modularity: 0.1486


Leiden modularity: 0.1486


## Louvain community stability

Repeated Louvain runs (co-assignment matrix), mean degree per community, and modularity across hierarchical levels of one run.

In [19]:
import importlib

importlib.reload(utils)

from utils import (
    louvain_communities_map_plot,
    plot_louvain_coassignment_heatmaps_pair,
    plot_louvain_community_mean_degrees,
    plot_louvain_community_node_counts,
    plot_louvain_modularity_trajectory,
    louvain_reference_partition,
    louvain_pre_final_communities_map_plot,
    louvain_modularity_by_level,
)

N_LOUVAIN_RUNS = 40
LOUVAIN_SEED = 42
HEATMAP_SAMPLE_SEED = 2024
MAX_HEATMAP_NODES = 40

# Reference partition (shared by map, heatmap order, bar colors, hover labels)
ref_communities, ref_modularity, ref_membership = louvain_reference_partition(
    G_undirected,
    seed=LOUVAIN_SEED,
)

# 0) Community map — use this to read Community 1, 2, … in the bar chart
map_fig, _, _, _ = louvain_communities_map_plot(
    G_undirected,
    communities=ref_communities,
    modularity_score=ref_modularity,
    seed=LOUVAIN_SEED,
    max_edges=900,
)
show(map_fig)
print(f"Reference Louvain modularity (final): {ref_modularity:.4f}")


# 1) Co-assignment heatmaps side by side (hover highlights row/column)
co_layout, co_matrices, co_label_groups, co_selected = plot_louvain_coassignment_heatmaps_pair(
    G_undirected,
    communities=ref_communities,
    membership=ref_membership,
    n_runs=N_LOUVAIN_RUNS,
    seed=LOUVAIN_SEED,
    max_nodes=MAX_HEATMAP_NODES,
    heatmap_sample_seed=HEATMAP_SAMPLE_SEED,
)
show(co_layout)
print(
    f"Left: {len(co_label_groups[0])} random high-degree airports | "
    f"Right: {len(co_label_groups[1])} balanced per community "
    f"({G_undirected.number_of_nodes()} nodes in graph)."
)


# 1b) Co-assignment heatmaps side by side without node limit
co_layout, co_matrices, co_label_groups, co_selected = plot_louvain_coassignment_heatmaps_pair(
    G_undirected,
    communities=ref_communities,
    membership=ref_membership,
    n_runs=N_LOUVAIN_RUNS,
    seed=LOUVAIN_SEED,
    max_nodes=10000,
    heatmap_sample_seed=HEATMAP_SAMPLE_SEED,
)
show(co_layout)
print(
    f"Left: {len(co_label_groups[0])} random high-degree airports | "
    f"Right: {len(co_label_groups[1])} balanced per community "
    f"({G_undirected.number_of_nodes()} nodes in graph)."
)

# 2) Mean weighted degree and node counts per community (same partition as map)
deg_fig, community_degrees = plot_louvain_community_mean_degrees(
    G_undirected,
    communities=ref_communities,
)
size_fig, _ = plot_louvain_community_node_counts(
    G_undirected,
    communities=ref_communities,
)
show(deg_fig)
show(size_fig)
display(community_degrees)

# 3) Modularity path: level 0 singletons (worst Q) → merges → final level (used)
mod_fig, modularity_path = plot_louvain_modularity_trajectory(
    G_undirected,
    seed=LOUVAIN_SEED,
)
show(mod_fig)
display(modularity_path)
final_q = modularity_path.loc[
    modularity_path["stage"] == "final (chosen partition)", "modularity"
].iloc[0]
pre_final_q = modularity_path.loc[
    modularity_path["stage"].str.contains("pre-final", na=False), "modularity"
]
print(
    f"Initial Q (singletons): {modularity_path.iloc[0]['modularity']:.4f} | "
    f"Pre-final Q: {pre_final_q.iloc[0]:.4f} | "
    f"Final Q (chosen): {final_q:.4f}"
)


# 0b) Pre-final hierarchical level (partition before the chosen final)
pre_final_fig, pre_final_modularity, _ = louvain_pre_final_communities_map_plot(
    G_undirected,
    seed=LOUVAIN_SEED,
    max_edges=900,
)
show(pre_final_fig)
print(f"Pre-final level modularity: {pre_final_modularity:.4f}")

Reference Louvain modularity (final): 0.1486


Left: 40 random high-degree airports | Right: 40 balanced per community (456 nodes in graph).


Left: 456 random high-degree airports | Right: 140 balanced per community (456 nodes in graph).


,community,n_nodes,mean_degree,color
0,Community 1,134,116.044776,#1f77b4
1,Community 2,153,177.666667,#ff7f0e
2,Community 3,35,161.342857,#2ca02c
3,Community 4,134,146.597015,#d62728


,level,modularity,n_communities,stage
0,0,-0.003301,456,initial (singletons)
1,1,0.147919,5,pre-final (level before final)
2,2,0.148641,4,final (chosen partition)
3,3,0.000000,1,"next level (over-merged, worse Q)"


Initial Q (singletons): -0.0033 | Pre-final Q: 0.1479 | Final Q (chosen): 0.1486


Pre-final level modularity: 0.1479
